# Baseball Pitcher Injury Predictor
This notebook provides an interactive tool for predicting injuries in Major League Baseball (MLB) pitchers using machine learning techniques.  The data includes MLB seasons from 2020 to 2024, predictions are not real-time.

Visualizations include:
* Upcoming Predicted Injuries for Philadelphia Phillies Pitchers
* Pitcher Search Injury Predictor
* Timeline of Features and Injuries
* Feature Importance

In [ ]:
# Import necessary libraries, global data and initialization. This may take a few minutes.

import ipywidgets as widgets
from datetime import datetime
from IPython.display import clear_output
from typing import List
from injury_prediction import InjuryPredictor, PredictionResult
from data.data_pitchers_with_predictions import available_pitchers
from data_preprocessing import g_x_test, g_y_test, g_x_train, g_y_train
import utils_model.visualizations as mv

# Initialize the injury predictor with the preprocessed data
injury_predictor = InjuryPredictor(g_x_train, g_y_train)

# Initialize model visualizations
visualizations = mv.ModelVisualizations(injury_predictor, g_x_test, g_y_test, "Baseball Injury Predictor")

In [ ]:
# Create Upcoming Predicted Injuries for Philadelphia Phillies Pitchers

# List of active Philadelphia Phillies pitchers
phillie_pitchers = ['Wheeler, Zack', 'Suárez, Ranger', 'Walker, Taijuan',
                    'Kerkering, Orion', 'Alvarado, José', 'Strahm, Matt', 'Ross, Joe']

# Function to get predictions for Phillies pitchers
def get_predictions() -> list[PredictionResult] | str:
    """Get predictions for pitchers and return a list"""
    predictions = []

    # Get predictions for each pitcher
    for pitcher in phillie_pitchers:
        result = injury_predictor.predict_next_injury(pitcher)
        predictions.append(result)

    # Check if predictions are empty
    if not predictions:
        return "No predictions available for Philadelphia pitchers."

    return predictions

table_html = f"""
    <table style='width: 1000px; border-collapse: collapse;'>
        <thead style='background-color: #3498db; color: white;'>
            <tr style='background-color: #3498db; color: white;'>
                <th style='padding: 10px; text-align: left;'>Pitcher Name</th>
                <th style='padding: 10px; text-align: left;'>Predicted Injury Date</th>
                <th style='padding: 10px; text-align: left;'>Days to Injury</th>
                <th style='padding: 10px; text-align: left;'>Most Recent Injury</th>
                <th style='padding: 10px; text-align: left;'>Current Pitch Count</th>
                <th style='padding: 10px; text-align: left;'>Most Common Pitch</th>
            </tr>
        </thead>
        <tbody>
"""
today = datetime.now()
today_str = today.strftime('%Y-%m-%d')

days_legend_html = f"""
    <p style='margin-top: 0;'>
        <span style='color: #e74c3c; font-weight: bold;'>Red</span> - Predicted injury in the past (overdue)<br>
        <span style='color: #f39c12; font-weight: bold;'>Orange</span> - Injury within 30 days<br>
        <span style='color: #2ecc71; font-weight: bold;'>Green</span> - Injury more than 30 days away
    </p>
"""
# Get predictions for Philadelphia pitchers
sorted_predictions = get_predictions()

# Check if the prediction is a string (error message)
if isinstance(sorted_predictions, str):
    table_html += """
        <tr>
            <td colspan='6' style='padding: 10px; text-align: center; color: red;'>No predictions available for Philadelphia pitchers.</td>
        </tr>
    """
else:
    sorted_predictions: List[PredictionResult] = sorted(
        sorted_predictions,
        key=lambda predict: predict.predicted_next_injury_date
    )

    # Add rows for each prediction
    for i, pred in enumerate(sorted_predictions):
        # Alternate row colors
        bg_color = 'white' if i % 2 == 0 else '#eaecee'

        # Calculate days from today (negative for past dates)
        days_from_today = (pred.predicted_next_injury_date - today).days
        days_text = f"{days_from_today} days"
        days_color = '#e74c3c' if days_from_today < 0 else '#2ecc71' if days_from_today > 30 else '#f39c12'

        table_html += f"""
            <tr style='background-color: {bg_color};'>
                <td style='padding: 10px; text-align: left;'>{pred.pitcher_name}</td>
                <td style='padding: 10px; text-align: left;'>{pred.predicted_next_injury_date.strftime('%Y-%m-%d')}</td>
                <td style='padding: 10px; text-align: left; color: {days_color}; font-weight: bold;'>{days_text}</td>
                <td style='padding: 10px; text-align: left;'>{pred.most_recent_injury.strftime('%Y-%m-%d')}</td>
                <td style='padding: 10px; text-align: left;'>{pred.current_pitch_count}</td>
                <td style='padding: 10px; text-align: left;'>{pred.most_common_pitch_type}</td>
            </tr>
        """

# Close the table
table_html += """
        </tbody>
    </table>
"""

cell_title_upcoming = f"""
    <h2>Upcoming Predicted Injuries for Philadelphia Phillies Pitchers</h2>
"""

# Create a container for all widgets
container_search = widgets.VBox([
    widgets.HTML(cell_title_upcoming),
    widgets.HTML(days_legend_html),
    widgets.HTML(table_html)
])

# Display the container
display(container_search)

In [ ]:
# Create Pitcher Search Injury Predictor

# Create a dropdown widget for selecting a pitcher
pitcher_dropdown = widgets.Dropdown(
    options=available_pitchers,
    description='Select Pitcher:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='33%')
)

# Create a button to trigger the prediction
predict_button = widgets.Button(
    description='Predict Next Injury',
    button_style='primary',
    tooltip='Click to predict the next injury for the selected pitcher',
    icon='check'
)

# Create an output widget to display the prediction results
output = widgets.Output()

def format_prediction_result(result):
    if isinstance(result, str):
        return f"<div style='color: red; font-weight: bold;'>{result}</div>"

    if isinstance(result, PredictionResult):
        html = f"""
            <h5 style='font-size: 14px; margin-bottom: 0'>Prediction Results for {result.pitcher_name}</h5>
            <table style='width: 500px; border-collapse: collapse; margin-bottom: 20px;'>
                <tr>
                    <td style='padding: 10px; font-weight: bold;'>Predicted next injury date:</td>
                    <td style='padding: 10px; color: #e74c3c; font-weight: bold;'>{result.predicted_next_injury_date.strftime('%Y-%m-%d')}</td>
                </tr>

                <tr style='background-color: #eaecee;'>
                    <td style='padding: 10px; font-weight: bold;'>Predicted days to next injury:</td>
                    <td style='padding: 10px;'>{result.predicted_days_to_next_injury:.1f} days</td>
                </tr>
                <tr>
                    <td style='padding: 10px; font-weight: bold;'>Most recent injury date:</td>
                    <td style='padding: 10px;'>{result.most_recent_injury.strftime('%Y-%m-%d')}</td>
                </tr>
                <tr style='background-color: #eaecee;'>
                    <td style='padding: 10px; font-weight: bold;'>Current pitch count:</td>
                    <td style='padding: 10px;'>{result.current_pitch_count}</td>
                </tr>
                <tr>
                    <td style='padding: 10px; font-weight: bold;'>Most common pitch type:</td>
                    <td style='padding: 10px;'>{result.most_common_pitch_type}</td>
                </tr>
                <tr style='background-color: #eaecee;'>
                    <td style='padding: 10px; font-weight: bold;'>Average effective speed:</td>
                    <td style='padding: 10px;'>{result.avg_effective_speed:.2f} mph</td>
                </tr>
                <tr>
                    <td style='padding: 10px; font-weight: bold;'>Average release spin rate:</td>
                    <td style='padding: 10px;'>{result.avg_release_spin_rate:.2f} rpm</td>
                </tr>
            </table>
        """
        return html

    return f"<div style='color: red;'>Unknown result type: {type(result)}</div>"

# Define the button click handler
def on_predict_button_clicked(b):
    with output:
        clear_output()

        # Get the selected pitcher
        selected_pitcher = pitcher_dropdown.value

        if selected_pitcher:
            # Make the prediction
            prediction_result = injury_predictor.predict_next_injury(selected_pitcher)

            # Display the prediction results
            display(widgets.HTML(format_prediction_result(prediction_result)))
        else:
            display(widgets.HTML("<div style='color: red;'>Please select a pitcher.</div>"))

# Attach the click handler to the button
predict_button.on_click(on_predict_button_clicked)

cell_title_search = f"""
    <h2>Pitcher Search Injury Predictor</h2>
"""

# Create a container for all widgets to ensure they display properly
container_pitcher_search = widgets.VBox([
    widgets.HTML(cell_title_search),
    widgets.HTML("<p style='margin-top: 0;'>Search for a pitcher by name and get a prediction for their next potential injury date.</p>"),
    widgets.HBox([pitcher_dropdown, predict_button]),
    output
])

# Display the container
display(container_pitcher_search)

In [ ]:
# Create Timeline of Features and Injuries

# Create a dropdown widget for selecting a pitcher
timeline_pitcher_dropdown = widgets.Dropdown(
    options=available_pitchers,
    description='Select Pitcher:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='33%')
)

# Create output widget for the visualization
timeline_output = widgets.Output()

def create_timeline_visualization(pitcher_name):
    """
    Create a timeline visualization for a selected pitcher showing:
    - Speed over time
    - Spin rate over time
    - Workload (pitch count) over time
    - Actual injuries
    - Predicted next injury

    Args:
        pitcher_name (str): Name of the pitcher to visualize
    """
    with timeline_output:
        timeline_output.clear_output()
        visualizations.features_injuries_timeline(pitcher_name)

# Create a button to trigger the visualization
visualize_button = widgets.Button(
    description='Generate Timeline',
    button_style='primary',
    tooltip='Click to generate the timeline visualization',
    icon='chart-line'
)

# Define the button click handler
def on_visualize_button_clicked(b):
    pitcher_name = timeline_pitcher_dropdown.value
    if pitcher_name:
        create_timeline_visualization(pitcher_name)
    else:
        with timeline_output:
            timeline_output.clear_output()
            display(widgets.HTML("<div style='color: red;'>Please select a pitcher.</div>"))

# Attach the click handler to the button
visualize_button.on_click(on_visualize_button_clicked)

# Add description text
timeline_description = widgets.HTML(
    """<p style='margin-top: 0;'>This visualization shows a timeline of key features (speed, spin rate, workload) for the selected pitcher.</p>
    <p>Red dashed lines indicate actual injuries, while purple dotted lines show predicted future injuries.</p>
    <p>Use the dropdown to select a pitcher, then click the 'Generate Timeline' button to create the visualization.</p>"""
)

cell_title_timeline = f"""
    <h2>Timeline of Features and Injuries</h2>
"""

# Create a container for all widgets
container_timeline = widgets.VBox([
    widgets.HTML(cell_title_timeline),
    timeline_description,
    widgets.HBox([timeline_pitcher_dropdown, visualize_button]),
    timeline_output
])

# Display the container
display(container_timeline)

In [ ]:
# Create Feature Importance

# Add description text
features_description = widgets.HTML(
    """<p style='margin-top: 0;'>This visualization shows the relative importance of different features in predicting pitcher injuries.
    Features with higher importance have a stronger influence on predictions.</p>"""
)

# Create output widget for the visualization
features_output = widgets.Output()

with features_output:
    features_output.clear_output()
    visualizations.feature_importance_pie_chart()

cell_title_features = f"""
    <h2>Feature Importance</h2>
"""

# Display the description
container_features = widgets.VBox([
    widgets.HTML(cell_title_features),
    features_description,
    features_output,
])

# Display the container
display(container_features)